In [ ]:
# Configuracao do ambiente (celula de setup do notebook)
import sys, os, warnings
sys.path.insert(0, os.path.abspath(os.path.join('..', '..', 'src')))
warnings.filterwarnings('ignore')
%matplotlib inline

# Capítulo 18 — ABRINDO A CAIXA-PRETA: FEATURE IMPORTANCE E SEUS LIMITES

> "Dizer que uma feature é 'importante' é como dizer que um jogador é 'importante'. É verdade, mas não conta o que ele fez no jogo de ontem: marcou o gol da vitória ou cometeu o pênalti da derrota?"
> — Um Cientista de Dados sobre importância global vs. local

Meu modelo estava calibrado, robusto e bem documentado. Mas a confiança não nasce de números: nasce do entendimento. Se eu quisesse que o chefe da oncologia confiasse no OncoClassify 2.0, teria de responder à pergunta mais fundamental: **"Por que o modelo tomou essa decisão?"**

Comecei pela visão panorâmica — quais *features* o modelo considera importantes *em geral*? Isso mostraria que ele presta atenção nas características certas: tamanho, forma, irregularidade do núcleo. Um começo necessário, mas, como eu logo perceberia, insuficiente.

## Interpretabilidade: Global vs. Local

Na Interpretabilidade de Machine Learning (XAI) há uma distinção crucial:

**Global** — explica o comportamento do modelo como um todo: "quais *features* mais importam, em média?" Ex.: Permutation Importance.

**Local** — explica uma *única previsão*: "para *este* paciente, o que pesou e quanto?" Ex.: SHAP, LIME (Capítulos 19 e 20).

Começamos pela global, com a **Permutation Importance**: embaralhamos os valores de uma *feature* (quebrando sua relação com o alvo) e medimos **quanto o desempenho cai**. Se cai muito, a *feature* era importante. É uma técnica **agnóstica ao modelo** e que captura interações.

#### Código 18.1: Importância por Permutação


In [ ]:
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.svm import SVC
from sklearn.inspection import permutation_importance
from oncoclassify_utils import X_train, y_train

def info_mutua(X, y):
    return mutual_info_classif(X, y, random_state=42)

modelo = Pipeline([("selecao", SelectKBest(info_mutua, k=25)),
                   ("escala", StandardScaler()),
                   ("svm", SVC(kernel="rbf", C=10, gamma=0.01, random_state=42))]).fit(X_train, y_train)

# Importancia medida no TREINO -- o cofre (X_test) permanece lacrado ate o
# Capitulo 24. Aqui nao estimamos desempenho, e sim de QUE features o modelo
# depende; para isso, o proprio treino serve. 15 embaralhamentos.
r = permutation_importance(modelo, X_train, y_train, n_repeats=15, random_state=42, n_jobs=-1)
imp = pd.Series(r.importances_mean, index=X_train.columns).sort_values(ascending=False)

print("Top 10 (queda média de acurácia ao embaralhar a feature):")
print(imp.head(10).round(4).to_string())
print(f"\nFeatures com efeito ~zero: {(imp.abs() <= 0.001).sum()} de 30")

Top 10 (queda média de acurácia ao embaralhar a feature):
worst concave points    0.0131
compactness error       0.0110
worst texture           0.0101
worst smoothness        0.0082
worst concavity         0.0075
worst area              0.0071
worst radius            0.0065
radius error            0.0065
worst symmetry          0.0057
area error              0.0056

Features com efeito ~zero: 7 de 30


![Permutation Importance](media/figura_18_1.png)

Consistente com tudo que vimos: `worst concave points` lidera, seguida de medidas de irregularidade e tamanho ("worst" de textura, suavidade, concavidade, área e raio). Embaralhá-las derruba a acurácia — sinal de que o modelo as usa de forma inteligente.

E as **7 *features* com efeito ~zero**? Cinco delas foram **descartadas pelo `SelectKBest(k=25)`** — o modelo literalmente não as vê, então embaralhá-las não muda nada. É um lembrete mecânico útil: a "importância" aqui é a importância *para este pipeline*, seleção incluída.

## Os Limites da Importância Global

Este gráfico é útil, mas limitado. Ele diz que, *em média*, `worst concave points` importa. Não diz: para *um* paciente específico, o valor daquela *feature* empurrou a previsão para maligno ou benigno? De quanto? Como interage com as outras?

Para responder isso, precisamos de interpretabilidade **local** — decompor uma única previsão em suas partes.

> **📌 CHECKLIST DO CAPÍTULO 18**
>
> ✅ Entende a diferença entre interpretabilidade **global** e **local**.
>
> ✅ Sabe como a **Permutation Importance** funciona e por que é agnóstica ao modelo.
>
> ✅ Executou o Código 18.1 e identificou as *features* mais críticas — e por que as descartadas pela seleção têm efeito zero.
>
> ✅ Compreende que a importância global diz *o que* importa, mas não *por que* uma previsão específica foi feita.
>
> **⚠️ CRÍTICO:** A importância global é um ótimo teste de sanidade: se o modelo apoiasse suas decisões numa *feature* irrelevante, você teria motivo para desconfiar, por mais alta que fosse a acurácia.

O gráfico confirmou que meu modelo não era um oráculo aleatório: ele focava nas características que os próprios patologistas valorizam. Eu podia dizer ao chefe da oncologia: "o modelo decide pelo tamanho, forma e irregularidade do núcleo".

Mas eu sabia qual seria a próxima pergunta dele: "*E para esta paciente aqui?* O modelo diz maligno com quase 99% de probabilidade. Por quê?" A importância global não respondia isso. Eu precisava de uma contabilidade exata para *cada* previsão. Precisava dos valores de SHAP.
